# Lesson 8.4 — Application Integration: SQS, SNS, Step Functions & EventBridge

**Level: Advanced** | **Exam:** SAP-C02 Domain 2, Tasks 2.4 & 2.5; Domain 4, Task 4.4

## Application Integration Services

| Service | Pattern | Use Case |
|---------|---------|---------|
| **SQS Standard** | Queue (pull, at-least-once) | High-throughput async processing |
| **SQS FIFO** | Queue (pull, exactly-once, ordered) | Financial transactions, ordering |
| **SNS** | Pub/Sub (push fan-out) | 1 message to N subscribers |
| **EventBridge** | Event bus with routing | Complex routing, SaaS, schema registry |
| **Step Functions** | State machine orchestration | Multi-step workflows, long-running |
| **Kinesis** | Streaming, ordered, replayable | Real-time analytics, log ingestion |

## SQS Key Settings
- **Visibility Timeout**: Time message is hidden while processing (set to max processing time)
- **DLQ (Dead Letter Queue)**: Where failed messages go (always configure this!)
- **Long Polling (WaitTimeSeconds=20)**: Reduces empty receive calls and cost
- **Message Retention**: 1 min to 14 days (default: 4 days)

## SNS Fan-out Pattern
```
OrderPlaced Event -> SNS Topic
    |-- SQS Queue -> Inventory Service
    |-- SQS Queue -> Shipping Service  
    |-- Lambda -> Email notification
    `-- SQS Queue -> Analytics pipeline
```

## Step Functions State Types
- **Task**: Invoke Lambda, ECS, DynamoDB, SNS, SQS, etc.
- **Choice**: Branch based on condition
- **Wait**: Pause for duration or until timestamp
- **Parallel**: Run multiple branches simultaneously
- **Map**: Iterate over an array (process each item)

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
import boto3
sqs = boto3.client('sqs', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')
sfn = boto3.client('stepfunctions', region_name='us-east-1')

# List SQS queues
queues = sqs.list_queues().get('QueueUrls', [])
print(f"SQS Queues ({len(queues)}):")
for url in queues[:5]:
    attrs = sqs.get_queue_attributes(QueueUrl=url, AttributeNames=['All'])['Attributes']
    name = url.split('/')[-1]
    msgs = attrs.get('ApproximateNumberOfMessages', '0')
    dlq = 'DLQ OK' if attrs.get('RedrivePolicy') else 'NO DLQ'
    print(f"  {name:35} Msgs:{msgs:5}  {dlq}")

# Step Functions example definition
print("\nStep Functions State Machine Example:")
order_workflow = {
    "Comment": "Order processing workflow",
    "StartAt": "ValidateOrder",
    "States": {
        "ValidateOrder": {"Type": "Task", "Resource": "arn:aws:lambda:::function:validate-order", "Next": "ProcessPayment"},
        "ProcessPayment": {"Type": "Task", "Resource": "arn:aws:lambda:::function:process-payment", 
                          "Catch": [{"ErrorEquals": ["PaymentFailed"], "Next": "NotifyFailure"}], "Next": "FulfillOrder"},
        "FulfillOrder": {"Type": "Parallel", "Branches": [
            {"StartAt": "UpdateInventory", "States": {"UpdateInventory": {"Type": "Task", "Resource": "arn:aws:lambda:::function:inventory", "End": True}}},
            {"StartAt": "ShipOrder", "States": {"ShipOrder": {"Type": "Task", "Resource": "arn:aws:lambda:::function:shipping", "End": True}}}
        ], "Next": "NotifySuccess"},
        "NotifySuccess": {"Type": "Task", "Resource": "arn:aws:states:::sns:publish", "Parameters": {"TopicArn": "arn:aws:sns:::OrderSuccess", "Message.$": "$.orderId"}, "End": True},
        "NotifyFailure": {"Type": "Task", "Resource": "arn:aws:states:::sns:publish", "Parameters": {"TopicArn": "arn:aws:sns:::OrderFailed"}, "End": True}
    }
}
import json
print(json.dumps(order_workflow, indent=2)[:500] + "...")

## Exercise

In [ ]:
# TODO: Implement a fan-out architecture
# 1. Create SNS topic "OrderEvents"
# 2. Create 2 SQS queues: "orders-inventory" and "orders-shipping"
# 3. Each queue should have a corresponding DLQ
# 4. Subscribe both queues to the SNS topic
# 5. Publish a test message to SNS and verify both queues receive it

import boto3, json
sqs = boto3.client('sqs', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']

# YOUR CODE HERE

## Summary
- **SQS**: Decouple producers and consumers; use DLQ to capture failed messages
- **SNS fan-out**: Distribute events to multiple consumers simultaneously
- **EventBridge**: Best for new architectures — rich routing, SaaS connectors, schema registry
- **Step Functions**: Visual workflows for multi-step processes with retry, catch, and parallel support
- **Pattern**: Event → EventBridge → SQS → Lambda = resilient, decoupled, auto-scaling

---
**Next:** [Phase 9 — Migration, Modernization & Continuous Improvement](../phase-9-migration/lesson-9.1-migration-assessment.ipynb)